In [0]:

from pyspark.sql import functions as F

hpi = spark.table("scottish_housing_ws.silver.uk_hpi")
awe = spark.table("scottish_housing_ws.silver.ons_awe")

In [0]:
hpi_scotland = (
    hpi
    .filter(F.col("area_code") == "S92000003")
    .select("report_date", "year_month", "region_name", "area_code", "average_price", "sales_volume")
)

In [0]:
gold = (
    hpi_scotland
    .join(awe.select("year_month", "avg_weekly_earnings_gbp"), on="year_month", how="inner")
    .withColumn(
        "affordability_ratio_weeks",
        F.round(F.col("average_price") / F.col("avg_weekly_earnings_gbp"), 1)
    )
    .withColumn(
        "affordability_ratio_years",
        F.round(F.col("average_price") / (F.col("avg_weekly_earnings_gbp") * F.lit(52)), 2)
    )
    .select(
        "report_date",
        "year_month",
        "region_name",
        "area_code",
        F.col("average_price").alias("nominal_average_price"),
        "avg_weekly_earnings_gbp",
        "affordability_ratio_weeks",
        "affordability_ratio_years",
        "sales_volume"
    )
    .orderBy("report_date")
)

In [0]:
print(f"Row count: {gold.count()}")
gold.orderBy("report_date").show(5)
gold.orderBy(F.col("report_date").desc()).show(5)

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.affordability_ratio")
)

In [0]:
(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_affordability_ratio/")
)

print("Gold 02 complete.")

In [0]:
# Sources: silver.uk_hpi, silver.ons_awe
# Grain: monthly, Scotland national
# Join key: year_month
#
# Affordability ratio = average house price / average weekly earnings
# Expressed as "number of weeks of average earnings to buy an average house"
# Higher = less affordable
#
# Limitation: ONS AWE is GB-wide, not Scotland-specific